# NAidioma — Fábrica de Lições em Lote no Kaggle (`inkling-2-mini`, GPU T4 x2)

**Regras comprovadas desta conta (leia antes de rodar):**
1. **Accelerator:** `GPU T4 x2` · **Internet:** `ON`.
2. **Rodar SEMPRE pelo navegador (`Edit → Run All`)** — envios pelo terminal ficam presos em `Queued` na fila de GPU.
3. **Somente fábrica de conteúdo em lote (JSON):** o Kaggle NÃO é usado como servidor 24h do app (sem túnel `cloudflared` em produção). Para o tutor ao vivo 24h, usamos **Ollama Cloud (`inkling-2-small`)** com fallback automático no **Gemini** (timeout 15s).
4. Baseado no notebook validado `andreajoa/nexacloud-inkling-2-mini` (~11–13 tokens/s, excelente em JSON).

In [ ]:
import os, subprocess, time, urllib.request

def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)
if not os.path.exists('/dev/nvidia0'):
    raise SystemExit('SEM GPU! Ative Accelerator: GPU T4 x2 e rode via navegador (Edit -> Run All).')

sh('apt-get -qq update >/dev/null && apt-get -qq install -y zstd pciutils lshw >/dev/null')
sh('curl -fsSL https://ollama.com/install.sh | sh')
sh('ollama --version || true')

env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_SCHED_SPREAD='1', OLLAMA_KEEP_ALIVE='-1')
log = open('/kaggle/working/ollama_serve.log', 'w')
subprocess.Popen(['ollama', 'serve'], env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        break
    except Exception:
        time.sleep(2)

MODEL = os.environ.get('NAIDIOMA_MODEL', 'inkling-2-mini')
t0 = time.time()
sh(f'ollama pull {MODEL}')
print(f'Modelo {MODEL} pronto em {time.time() - t0:.0f}s!', flush=True)

In [ ]:
import json, time, urllib.request

SYSTEM_PROMPT = (
    'Você é um especialista pedagógico do app NAidioma (estilo Duolingo, amigável para crianças de 9 anos e adultos). '
    'Responda SOMENTE JSON válido compatível com o schema LessonSchema de @naidioma/shared. '
    'Use português brasileiro claro, curto e alegre.'
)

def generate_lesson_batch(lang_code, lang_name, topic_pt, cefr='A1'):
    user_prompt = f'''Gere uma lição JSON para o app NAidioma ({lang_name}, código "{lang_code}").
Tópico: {topic_pt}. Nível CEFR: {cefr}.
Inclua exatamente 4 exercícios variados usando os tipos: picture_choice (com emoji em cada option), match_pairs (4 pares), listen_build e speak_pronounce.
Para Coreano/Japonês/Chinês, inclua sempre reading (romanização) e ipaOrPhoneticBr.
Responda APENAS JSON válido.'''
    body = {
        'model': MODEL,
        'stream': False,
        'format': 'json',
        'options': {'num_ctx': 8192, 'temperature': 0.35},
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    }
    req = urllib.request.Request(
        'http://127.0.0.1:11434/api/chat',
        data=json.dumps(body).encode(),
        headers={'Content-Type': 'application/json'},
    )
    t = time.time()
    raw = json.loads(urllib.request.urlopen(req, timeout=600).read())
    tok = raw.get('eval_count', 0)
    dur = raw.get('eval_duration', 1) / 1e9
    lesson = json.loads(raw['message']['content'])
    print(f'[{lang_code}] {topic_pt} -> {round(time.time()-t,1)}s ({round(tok/dur,1) if dur else 0} tok/s)', flush=True)
    return lesson

BATCH_TOPICS = [
    ('ko', 'Coreano', 'Rotina da Dona de Casa: Cozinha, Lavar Louça e Mercado', 'A1'),
    ('ko', 'Coreano', 'Rotina da Criança e do Aluno: Escola, Mochila e Brincadeiras', 'A1'),
    ('en', 'Inglês para Brasileiros', 'Rotina do Trabalhador: Ônibus, Almoço e Celular', 'A1'),
    ('en', 'Inglês para Brasileiros', 'Rotina em Casa: Cozinha, Compras e Família', 'A1'),
]

out = []
for code, name, topic, cefr in BATCH_TOPICS:
    try:
        out.append(generate_lesson_batch(code, name, topic, cefr))
    except Exception as exc:
        print(f'Erro em {code}/{topic}:', exc, flush=True)

with open('/kaggle/working/naidioma_batch_lessons.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(f'Lote concluído: {len(out)} lições salvas em /kaggle/working/naidioma_batch_lessons.json', flush=True)